<a href="https://colab.research.google.com/github/ghadirchhade/Master-Thesis/blob/main/baseline2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

In [ ]:
!pip install -q torch torchvision

In [ ]:
import torch
import torchvision

print("PyTorch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)
print("CUDA is available:", torch.cuda.is_available())

PyTorch version: 2.11.0+cu128
Torchvision version: 0.26.0+cu128
CUDA is available: True


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# facebook/sam3 is gated on the Hub -> you must accept the license at
# https://huggingface.co/facebook/sam3 with the account whose token you use below.
!pip install -q -U transformers accelerate huggingface_hub supervision

from huggingface_hub import login
login()  # paste your HF token (needs access to facebook/sam3)

print("Transformers SAM3 dependencies installed.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 76.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 774.9/774.9 kB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.2/280.2 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.7/102.7 kB 5.1 MB/s eta 0:00:00
Transformers SAM3 dependencies installed.


In [ ]:
import os, glob, csv, time
import numpy as np
import pandas as pd
import torch
import cv2
from PIL import Image
import supervision as sv
import matplotlib.patches as patches
from transformers import Sam3Model, Sam3Processor
from supervision.metrics import MeanAveragePrecision

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

sam3_model = Sam3Model.from_pretrained("facebook/sam3", device_map="auto")
sam3_model.eval()

sam3_processor = Sam3Processor.from_pretrained("facebook/sam3")

print("HF transformers SAM3 model + processor loaded.")
print("Model device:", next(sam3_model.parameters()).device)

config.json:   0%|          | 0.00/25.8k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.44GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/1468 [00:00<?, ?it/s]

processor_config.json:   0%|          | 0.00/1.71k [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.64M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/588 [00:00<?, ?B/s]

HF transformers SAM3 model + processor loaded.
Model device: cuda:0


In [ ]:
#configuration
IMAGES_ROOT = "/content/drive/MyDrive/master_thesis/dataset/AGS_Multi_Rumex/images"
LABELS_ROOT = "/content/drive/MyDrive/master_thesis/dataset/AGS_Multi_Rumex/annotations_yolo"
RUMEX_CLASS_ID = 0

EXPERIMENT_NAME = "E02_1"
N_EXEMPLARS     = 3               # several positive bboxes
MAX_DIM         = 1024            # resize cap for SAM3 input (from your Exp2 code)
THRESHOLD       = 0.3             # confidence threshold
MASK_THRESHOLD  = 0.5             # from your Exp2 code (Exp1 used 0.4 -- kept as-is per experiment)
IOU_THRESHOLD   = 0.5

PROMPT_TYPE = "multiple" if N_EXEMPLARS > 1 else "single"

OUTPUT_DIR = "/content/drive/MyDrive/master_thesis/results/"
os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_CSV = os.path.join(OUTPUT_DIR, f"results_{EXPERIMENT_NAME}.csv")
CSV_COLUMNS = ["experiment_name", "image_ID", "Prompt_ID", "Prompt_Type", "mAP50",
               "precision", "recall", "IoU1", "IoU2"]

RNG = np.random.default_rng(42)  # same seed as Exp1 -> same anchor/exemplar sampling logic

In [ ]:
#  Dataset discovery
def find_label_path(image_filename_no_ext, folder_name):
    mirrored = os.path.join(LABELS_ROOT, folder_name, image_filename_no_ext + ".txt")
    flat = os.path.join(LABELS_ROOT, image_filename_no_ext + ".txt")
    if os.path.exists(mirrored):
        return mirrored
    if os.path.exists(flat):
        return flat
    return None

image_records = []
VALID_EXT = (".jpg", ".jpeg", ".png")

for folder in sorted(os.listdir(IMAGES_ROOT)):
    folder_path = os.path.join(IMAGES_ROOT, folder)
    if not os.path.isdir(folder_path):
        continue
    for fname in sorted(os.listdir(folder_path)):
        if not fname.lower().endswith(VALID_EXT):
            continue
        name_no_ext = os.path.splitext(fname)[0]
        label_path = find_label_path(name_no_ext, folder)
        image_id = f"{folder}/{name_no_ext}"
        image_records.append((folder, os.path.join(folder_path, fname), label_path, image_id))

missing_labels = [r for r in image_records if r[2] is None]
print(f"Discovered {len(image_records)} images across {len(set(r[0] for r in image_records))} folders.")
if missing_labels:
    print(f"WARNING: {len(missing_labels)} images have no matching label file. "
          f"First few: {[r[3] for r in missing_labels[:5]]}")

Discovered 179 images across 15 folders.


In [ ]:
#  YOLO box loader + exemplar/prompt-id helpers
def load_yolo_boxes(label_path, img_width, img_height, class_id=0):
    boxes = []
    with open(label_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue
            cls = int(parts[0])
            if cls != class_id:
                continue
            xc, yc, bw, bh = map(float, parts[1:5])
            xc, yc, bw, bh = xc * img_width, yc * img_height, bw * img_width, bh * img_height
            boxes.append([xc - bw / 2, yc - bh / 2, xc + bw / 2, yc + bh / 2])
    return np.array(boxes, dtype=np.float32)


def select_exemplar_indices(n_gt: int, anchor_idx: int, n_exemplars: int, rng) -> list:
    """
    Always includes the anchor box; fills the remaining slots with other GT
    boxes from the same image, sampled without replacement. Falls back to
    whatever's available if the image doesn't have enough other boxes.
    """
    others = [i for i in range(n_gt) if i != anchor_idx]
    n_others_needed = min(n_exemplars - 1, len(others))
    chosen_others = list(rng.choice(others, size=n_others_needed, replace=False)) if n_others_needed > 0 else []
    return [anchor_idx] + chosen_others


def format_prompt_id(exemplar_indices: list) -> str:
    """Anchor first, then sampled others, e.g. '5+12+3' -- anchor is always the first number."""
    return "+".join(str(i) for i in exemplar_indices)

In [ ]:
#  Resize helper
def resize_for_sam3(img, max_dim=MAX_DIM):
    w, h = img.size
    scale = max_dim / max(w, h)
    if scale >= 1:
        return img, 1.0
    new_w, new_h = int(w * scale), int(h * scale)
    return img.resize((new_w, new_h), Image.BILINEAR), scale

In [ ]:
# # ============================================================
# # CELL: Direct scale-mismatch check
# # ============================================================
# folder, image_path, label_path, image_id = valid_images[0]
# image = Image.open(image_path).convert("RGB")
# img_w, img_h = image.size
# gt_boxes = load_yolo_boxes(label_path, img_w, img_h, class_id=RUMEX_CLASS_ID)

# image_sam, sam_scale = resize_for_sam3(image, MAX_DIM)
# print(f"Full-res image size: {image.size}")
# print(f"image_sam size (resized): {image_sam.size}")
# print(f"sam_scale: {sam_scale}")

# anchor_idx = 0
# exemplar_indices = select_exemplar_indices(len(gt_boxes), anchor_idx, N_EXEMPLARS, RNG)
# exemplar_boxes_fullres = [gt_boxes[i].tolist() for i in exemplar_indices]
# print(f"\nExemplar boxes (full-res, these ARE gt_boxes): {exemplar_boxes_fullres}")

# # Reproduce run_sam3_whole_image's internals step by step, with prints at each stage
# input_boxes_xyxy = [[c * sam_scale for c in box] for box in exemplar_boxes_fullres]
# print(f"Exemplar boxes scaled to image_sam space (sent to SAM3): {input_boxes_xyxy}")

# inputs = sam3_processor(
#     images=image_sam,
#     input_boxes=[input_boxes_xyxy],
#     input_boxes_labels=[[1] * len(exemplar_boxes_fullres)],
#     return_tensors="pt",
# ).to(sam3_model.device)

# print(f"\ninputs['original_sizes']: {inputs.get('original_sizes')}")
# print(f"(compare this to image_sam size {image_sam.size} -- should match [H,W] or [W,H])")

# with torch.no_grad():
#     outputs = sam3_model(**inputs)

# results = sam3_processor.post_process_instance_segmentation(
#     outputs, threshold=THRESHOLD, mask_threshold=MASK_THRESHOLD,
#     target_sizes=inputs.get("original_sizes").tolist(),
# )[0]

# pred_boxes_raw = to_numpy(results["boxes"])
# print(f"\nPredicted boxes BEFORE /sam_scale (raw from post_process): \n{pred_boxes_raw}")
# print(f"Range: min={pred_boxes_raw.min() if len(pred_boxes_raw) else 'N/A'}, "
#       f"max={pred_boxes_raw.max() if len(pred_boxes_raw) else 'N/A'}")

# pred_boxes_scaled = pred_boxes_raw / sam_scale
# print(f"\nPredicted boxes AFTER /sam_scale: \n{pred_boxes_scaled}")
# print(f"Range: min={pred_boxes_scaled.min() if len(pred_boxes_scaled) else 'N/A'}, "
#       f"max={pred_boxes_scaled.max() if len(pred_boxes_scaled) else 'N/A'}")

# print(f"\nFor comparison, exemplar/GT boxes (full-res): {exemplar_boxes_fullres}")
# print(f"GT box range: min={gt_boxes.min()}, max={gt_boxes.max()}")

In [ ]:
# pred_scores_np = to_numpy(results["scores"])

# print("=== Inputs to compute_detection_metrics ===")
# print("pred_boxes_scaled:", pred_boxes_scaled)
# print("pred_scores_np:", pred_scores_np)
# print("gt_boxes:", gt_boxes)

# # Temporarily expose the real exception instead of swallowing it
# import traceback

# pred_detections = sv.Detections(
#     xyxy=pred_boxes_scaled,
#     confidence=pred_scores_np,
#     class_id=np.zeros(len(pred_scores_np), dtype=int),
# )
# gt_detections = sv.Detections(
#     xyxy=gt_boxes,
#     class_id=np.zeros(len(gt_boxes), dtype=int),
# )

# try:
#     map_metric = MeanAveragePrecision()
#     result = map_metric.update([pred_detections], [gt_detections]).compute()
#     print("map50 SUCCESS:", result.map50)
# except Exception:
#     print("map50 THREW AN EXCEPTION:")
#     traceback.print_exc()

# # Now run your own IoU-matching logic directly, bypassing sv entirely
# metrics = compute_detection_metrics(pred_boxes_scaled, pred_scores_np, gt_boxes, iou_threshold=IOU_THRESHOLD)
# print("compute_detection_metrics output:", metrics)

In [ ]:
#  SAM3 whole-image (no tiling) inference, exemplar boxes as prompts
def to_numpy(x):
    """Handles tensors (incl. bf16/fp16), lists of tensors, or plain arrays."""
    if torch.is_tensor(x):
        if x.dtype in (torch.bfloat16, torch.float16):
            x = x.float()
        return x.detach().cpu().numpy()
    if isinstance(x, (list, tuple)):
        if len(x) > 0 and torch.is_tensor(x[0]):
            x = [t.float() if t.dtype in (torch.bfloat16, torch.float16) else t for t in x]
            return torch.stack([t.detach().cpu() for t in x]).numpy()
        return np.array(x)
    return np.array(x)


def run_sam3_whole_image(image: Image.Image, exemplar_boxes_fullres: list,
                          threshold: float = THRESHOLD, mask_threshold: float = MASK_THRESHOLD):
    """
    exemplar_boxes_fullres: list of [x1,y1,x2,y2] in the ORIGINAL image's pixel coords.

    Mirrors your Exp2 logic exactly:
      - resize whole image to MAX_DIM
      - scale exemplar boxes into that resized space
      - single SAM3 forward pass, no tiling
      - scale predicted boxes back up to full-res, upsample masks to full-res

    Returns: pred_boxes_fullres (np.ndarray Nx4), pred_scores (np.ndarray N),
             pred_masks_fullres (np.ndarray N,H,W bool) or None if no detections
    """
    img_w, img_h = image.size
    image_sam, sam_scale = resize_for_sam3(image, MAX_DIM)

    input_boxes_xyxy = [[c * sam_scale for c in box] for box in exemplar_boxes_fullres]
    input_boxes = [input_boxes_xyxy]
    input_boxes_labels = [[1] * len(exemplar_boxes_fullres)]

    inputs = sam3_processor(
        images=image_sam,
        input_boxes=input_boxes,
        input_boxes_labels=input_boxes_labels,
        return_tensors="pt",
    ).to(sam3_model.device)

    with torch.no_grad():
        outputs = sam3_model(**inputs)

    results = sam3_processor.post_process_instance_segmentation(
        outputs, threshold=threshold, mask_threshold=mask_threshold,
        target_sizes=inputs.get("original_sizes").tolist(),
    )[0]

    print("Returned keys:", results.keys())
    print("Boxes:", len(results["boxes"]))
    print("Scores:", results["scores"])

    pred_boxes_np = to_numpy(results["boxes"])
    pred_scores_np = to_numpy(results["scores"])
    pred_masks_np = to_numpy(results["masks"])

    # Boxes/masks came back in image_sam's (resized) coordinate space -> rescale to full-res
    if len(pred_boxes_np) > 0:
        pred_boxes_np = pred_boxes_np / sam_scale

    full_res_masks = None
    if len(pred_masks_np) > 0:
        full_res_masks = np.zeros((len(pred_masks_np), img_h, img_w), dtype=bool)
        for i, m in enumerate(pred_masks_np):
            m_resized = cv2.resize(m.astype(np.uint8), (img_w, img_h), interpolation=cv2.INTER_NEAREST)
            full_res_masks[i] = m_resized.astype(bool)

    return pred_boxes_np, pred_scores_np, full_res_masks

In [ ]:
# Metrics (identical logic/formulas to Exp1 -- mAP50, precision,
# recall, IoU1 = matched-only mean IoU, IoU2 = mean IoU over all GT)

# matrix of IoU between every pred and every GT
def compute_iou_matrix(boxes1, boxes2):
    if len(boxes1) == 0 or len(boxes2) == 0: #if there r no pred or no GTs => no comparison
        return np.zeros((len(boxes1), len(boxes2)))
    #each box is stored as x1,y1,x2,y2
    x1 = np.maximum(boxes1[:, None, 0], boxes2[None, :, 0])  #intersection of left boundary
    y1 = np.maximum(boxes1[:, None, 1], boxes2[None, :, 1])  #intersection of top boundary
    x2 = np.minimum(boxes1[:, None, 2], boxes2[None, :, 2])  #intersection of right boundary
    y2 = np.minimum(boxes1[:, None, 3], boxes2[None, :, 3])  #intersection of bottom boundary
    inter_w = np.clip(x2 - x1, 0, None)
    inter_h = np.clip(y2 - y1, 0, None)
    inter_area = inter_w * inter_h #rectangle area
    area1 = (boxes1[:, 2] - boxes1[:, 0]) * (boxes1[:, 3] - boxes1[:, 1]) # area of every predicted box(height x width)
    area2 = (boxes2[:, 2] - boxes2[:, 0]) * (boxes2[:, 3] - boxes2[:, 1])
    union_area = area1[:, None] + area2[None, :] - inter_area #prediction+ GT - intersection
    return np.where(union_area > 0, inter_area / union_area, 0.0)


def compute_detection_metrics(pred_boxes_np, pred_scores_np, gt_boxes, iou_threshold: float = 0.5) -> dict:
    pred_detections = sv.Detections(
        xyxy=pred_boxes_np if len(pred_boxes_np) else np.zeros((0, 4), dtype=np.float32),
        confidence=pred_scores_np if len(pred_scores_np) else np.zeros((0,), dtype=np.float32),
        class_id=np.zeros(len(pred_scores_np), dtype=int),
    )
    gt_detections = sv.Detections(
        xyxy=gt_boxes,
        class_id=np.zeros(len(gt_boxes), dtype=int),
    )
    try:
        map_metric = MeanAveragePrecision() #creating an empty evaluator
        result = map_metric.update([pred_detections], [gt_detections]).compute() #update it
        map50 = float(result.map50)
    except Exception:
        map50 = 0.0 if len(gt_boxes) > 0 else float("nan")

    num_preds, num_gt = len(pred_boxes_np), len(gt_boxes)
    iou_matrix = compute_iou_matrix(pred_boxes_np, gt_boxes)

    matched_gt = set() #stores GT boxes already matches (prevent matching multiple preds to the same GT)
    true_positives = 0  #each successful match increases it
    matched_ious = []  #stores IoU of successful matches
    pred_order = np.argsort(-pred_scores_np) if num_preds > 0 else [] #sort predictions by:
    #(adding minus sign to each pred score then np.argsort returns the indices that would sort the array in ascending manner then evaluate the predictions based on the indices
    #highest confidence first)

    for pred_idx in pred_order: #loop over predictions
        if num_gt == 0:
            break #nothing to match
        best_gt_idx = np.argmax(iou_matrix[pred_idx]) # take the row of the current pred_idx with all GTs then choose the largest index of IoU
        best_iou = iou_matrix[pred_idx, best_gt_idx]  #get IoU value
        if best_iou >= iou_threshold and best_gt_idx not in matched_gt: #(2nd condition: 1Gt => 1 prediction)
            matched_gt.add(best_gt_idx)
            true_positives += 1
            matched_ious.append(best_iou)

    precision = true_positives / num_preds if num_preds > 0 else 0.0
    recall = true_positives / num_gt if num_gt > 0 else 0.0
    iou1_matched_only = float(np.mean(matched_ious)) if matched_ious else 0.0
    iou2_over_all_gt = float(np.sum(matched_ious) / num_gt) if num_gt > 0 else 0.0

    return {
        "map50": map50, "precision": precision, "recall": recall,
        "iou_matched": iou1_matched_only, "iou_all_gt": iou2_over_all_gt,
    }

In [ ]:
#  Main automated loop -- Exp2 (no tiling), per-run + per-image logging
done_keys = set()
file_exists = os.path.exists(OUTPUT_CSV)
if file_exists:
    existing = pd.read_csv(OUTPUT_CSV)
    existing = existing[existing["experiment_name"] == EXPERIMENT_NAME]
    done_keys = set(zip(existing["image_ID"], existing["Prompt_ID"].astype(str)))
    print(f"Resuming: {len(done_keys)} rows already done for {EXPERIMENT_NAME}.")

csv_file = open(OUTPUT_CSV, "a", newline="")
csv_writer = csv.DictWriter(csv_file, fieldnames=CSV_COLUMNS)
if not file_exists:
    csv_writer.writeheader()

start_time = time.time()
n_runs = 0
image_times = []

#Only use images that have YOLO annotation files (third dimension is the label path)
valid_images = [rec for rec in image_records if rec[2] is not None]
n_total_images = len(valid_images)

#loop through every img
for img_idx, (folder, image_path, label_path, image_id) in enumerate(valid_images, start=1):
    image_t0 = time.time()

    image = Image.open(image_path).convert("RGB") #load img
    img_w, img_h = image.size #load dimensions
    gt_boxes = load_yolo_boxes(label_path, img_w, img_h, class_id=RUMEX_CLASS_ID)
    n_gt = len(gt_boxes)

    if n_gt == 0: #skip images without annotations
        print(f"[{EXPERIMENT_NAME}] ({img_idx}/{n_total_images}) {image_id}: 0 GT boxes, skipped.")
        continue

    image_map50s = [] #to store mAP50 values for this img (all prompt results for 1 img)

    for anchor_idx in range(n_gt): #each GT bbox become a positive input bbox for SAM3
        exemplar_indices = select_exemplar_indices(n_gt, anchor_idx, N_EXEMPLARS, RNG)
        prompt_id = format_prompt_id(exemplar_indices)

        if (image_id, prompt_id) in done_keys: #skip completed runs (already evaluated and stored in csv)
            continue

        run_t0 = time.time()

        #create prompt boxes for SAM3
        exemplar_boxes_fullres = [gt_boxes[i].tolist() for i in exemplar_indices]

        #run SAM3
        pred_boxes_np, pred_scores_np, _ = run_sam3_whole_image(
            image, exemplar_boxes_fullres, threshold=THRESHOLD, mask_threshold=MASK_THRESHOLD
        )

        #calculate metrics
        metrics = compute_detection_metrics(pred_boxes_np, pred_scores_np, gt_boxes, iou_threshold=IOU_THRESHOLD)
        image_map50s.append(metrics["map50"])

        #save results to csv
        row = {
            "experiment_name": EXPERIMENT_NAME,
            "image_ID": image_id,
            "Prompt_ID": prompt_id,
            "Prompt_Type": PROMPT_TYPE,
            "mAP50": metrics["map50"],
            "precision": metrics["precision"],
            "recall": metrics["recall"],
            "IoU1": metrics["iou_matched"],
            "IoU2": metrics["iou_all_gt"],
        }
        csv_writer.writerow(row)
        csv_file.flush()
        n_runs += 1

        run_elapsed = time.time() - run_t0
        #this print is executed for every run in an image (let's say we have 5 GTs so this print
        #is executed 5 times at each time the anchor is 1 of the GTs and the other 2 are sampled randomly)
        print(
            f"    [{EXPERIMENT_NAME}] run #{n_runs} | image={image_id} | "
            f"anchor={anchor_idx} ({anchor_idx+1}/{n_gt}) | prompt_id={prompt_id} | "
            f"mAP50={metrics['map50']:.3f} | time={run_elapsed:.1f}s"
        )

        torch.cuda.empty_cache()

    image_elapsed = time.time() - image_t0
    image_times.append(image_elapsed)
    avg_time_per_image = np.mean(image_times)
    images_left = n_total_images - img_idx
    eta_seconds = images_left * avg_time_per_image

    map50_str = f"{np.mean(image_map50s):.3f}" if image_map50s else "N/A (all anchors already done, skipped)"

    #this print will run finally after finishing all runs per image
    print(
        f"[{EXPERIMENT_NAME}] ({img_idx}/{n_total_images}) {image_id} done | "
        f"{n_gt} GT box(es) | image_mAP50_mean={map50_str} | "
        f"time={image_elapsed:.1f}s | avg/image={avg_time_per_image:.1f}s | "
        f"ETA={eta_seconds/60:.1f} min ({eta_seconds/3600:.2f} h)"
    )

csv_file.close()
total_elapsed = time.time() - start_time
print(f"\nFinished {EXPERIMENT_NAME}: {n_runs} new rows written to {OUTPUT_CSV}")
print(f"Total time: {total_elapsed/60:.1f} min ({total_elapsed/3600:.2f} h)")

Streaming output truncated to the last 5000 lines.
Boxes: 57
Scores: tensor([0.5797, 0.5129, 0.4121, 0.3093, 0.5148, 0.6246, 0.4044, 0.3727, 0.4410,
        0.4413, 0.4523, 0.6266, 0.5340, 0.4246, 0.4400, 0.3284, 0.3831, 0.5466,
        0.3177, 0.5671, 0.3960, 0.4907, 0.4211, 0.3726, 0.4057, 0.6503, 0.3027,
        0.4824, 0.5990, 0.4637, 0.4669, 0.3474, 0.3910, 0.6478, 0.5540, 0.5034,
        0.3889, 0.3530, 0.5653, 0.5357, 0.3058, 0.8882, 0.5300, 0.3767, 0.5815,
        0.6487, 0.6289, 0.3613, 0.6309, 0.8923, 0.3598, 0.4357, 0.3667, 0.5825,
        0.3996, 0.3890, 0.4134], device='cuda:0')
    [E02_1] run #1029 | image=20230426_Wallenwil/DJI_20230426111258_0224 | anchor=40 (41/55) | prompt_id=40+20+17 | mAP50=0.254 | time=6.5s
Returned keys: dict_keys(['scores', 'boxes', 'masks'])
Boxes: 57
Scores: tensor([0.6233, 0.6173, 0.3684, 0.6347, 0.5185, 0.5527, 0.5417, 0.5495, 0.5792,
        0.8389, 0.4213, 0.7556, 0.7059, 0.3342, 0.7337, 0.6879, 0.3502, 0.4299,
        0.5856, 0.3398, 0.53

In [ ]:
file_exists = os.path.exists(OUTPUT_CSV)
print(file_exists)

True


In [ ]:
# Per-experiment statistical summary
EXPERIMENT_CSV = OUTPUT_CSV
df = pd.read_csv(EXPERIMENT_CSV)
exp_name = df["experiment_name"].iloc[0]


exp_name = df["experiment_name"].iloc[0]
print(f"Experiment: {exp_name}")
print(f"Total rows (image x prompt runs): {len(df)}")
print(f"Number of distinct images: {df['image_ID'].nunique()}")

Experiment: E02_1
Total rows (image x prompt runs): 1841
Number of distinct images: 136


In [ ]:
# Raw per-run stats (every row = one image + one prompt set, independent observation)
# Here, every row in our CSV is treated as one independent experiment.
# output 1 value per metric (produces one overall mean and one overall std for each metric)
raw_summary = df.agg(
    n_runs=("mAP50", "count"),
    mAP50_mean=("mAP50", "mean"),
    mAP50_std=("mAP50", "std"),
    precision_mean=("precision", "mean"),
    precision_std=("precision", "std"),
    recall_mean=("recall", "mean"),
    recall_std=("recall", "std"),
    IoU1_mean=("IoU1", "mean"),
    IoU1_std=("IoU1", "std"),
    IoU2_mean=("IoU2", "mean"),
    IoU2_std=("IoU2", "std"),
)
raw_summary_path = os.path.join(OUTPUT_DIR, f"raw_summary_{exp_name}.csv")
raw_summary.to_csv(raw_summary_path)
print(f"Saved raw_summary_{exp_name}.csv to {OUTPUT_DIR}")

print(f"=== {exp_name} -- raw per-run summary (all rows independent) ===")
print(raw_summary)

Saved raw_summary_E02_1.csv to /content/drive/MyDrive/master_thesis/results/
=== E02_1 -- raw per-run summary (all rows independent) ===
                      mAP50  precision    recall      IoU1      IoU2
n_runs          1841.000000        NaN       NaN       NaN       NaN
mAP50_mean         0.288894        NaN       NaN       NaN       NaN
mAP50_std          0.245963        NaN       NaN       NaN       NaN
precision_mean          NaN   0.407793       NaN       NaN       NaN
precision_std           NaN   0.188216       NaN       NaN       NaN
recall_mean             NaN        NaN  0.349172       NaN       NaN
recall_std              NaN        NaN  0.241010       NaN       NaN
IoU1_mean               NaN        NaN       NaN  0.753743       NaN
IoU1_std                NaN        NaN       NaN  0.085388       NaN
IoU2_mean               NaN        NaN       NaN       NaN  0.272232
IoU2_std                NaN        NaN       NaN       NaN  0.210767


In [ ]:
# Image-level stats (collapse multiple prompt runs per image to one value first,
# so images with more GT boxes / more prompt sets don't dominate the average)
image_level = ( # produces one mean value per image for each metric
    df.groupby("image_ID")
    .agg(
        mAP50_image_mean=("mAP50", "mean"),
        precision_image_mean=("precision", "mean"),
        recall_image_mean=("recall", "mean"),
        IoU1_image_mean=("IoU1", "mean"),
        IoU2_image_mean=("IoU2", "mean"),
        n_prompts=("mAP50", "count"),
    )
    .reset_index()
)

image_level_path = os.path.join(OUTPUT_DIR, f"image_level_{exp_name}.csv")
image_level.to_csv(image_level_path, index=False)
print(f"Saved image-level {exp_name}.csv to {OUTPUT_DIR}")

print(f"=== {exp_name} -- image-level results ({len(image_level)} images) ===")
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    print(image_level.to_string(index=False))

Saved image-level E02_1.csv to /content/drive/MyDrive/master_thesis/results/
=== E02_1 -- image-level results (136 images) ===
                                    image_ID  mAP50_image_mean  precision_image_mean  recall_image_mean  IoU1_image_mean  IoU2_image_mean  n_prompts
     20220513_Halden/DJI_20220513075116_0213          1.000000              1.000000           1.000000         0.931989         0.931989          1
     20220513_Halden/DJI_20220513075207_0234          1.000000              0.666667           1.000000         0.918143         0.918143          2
     20220513_Halden/DJI_20220513075336_0270          0.767111              0.317844           0.833333         0.831875         0.692201          6
     20220513_Halden/DJI_20220513075432_0295          1.000000              0.600000           1.000000         0.869406         0.869406          3
     20220513_Halden/DJI_20220513075444_0300          1.000000              0.600000           1.000000         0.923465        

In [ ]:
#One overall mean + one overall std across all images
#at the level of all image-level
#every image contributes exactly once (gives equal weight to every image)
image_level_summary = pd.DataFrame({
    "experiment_name": [exp_name],
    "n_images": [image_level["image_ID"].nunique()],
    "mAP50_mean": [image_level["mAP50_image_mean"].mean()],
    "mAP50_std": [image_level["mAP50_image_mean"].std()],
    "precision_mean": [image_level["precision_image_mean"].mean()],
    "precision_std": [image_level["precision_image_mean"].std()],
    "recall_mean": [image_level["recall_image_mean"].mean()],
    "recall_std": [image_level["recall_image_mean"].std()],
    "IoU1_mean": [image_level["IoU1_image_mean"].mean()],
    "IoU2_mean": [image_level["IoU2_image_mean"].mean()],
})

image_level_summary.to_csv(os.path.join(OUTPUT_DIR, f"summary_{exp_name}.csv"), index=False)
print(f"\nSaved image_level_{exp_name}.csv and summary_{exp_name}.csv to {OUTPUT_DIR}")

print(f"\n=== {exp_name} -- image-level summary ===")
print(image_level_summary.to_string(index=False))


Saved image_level_E02_1.csv and summary_E02_1.csv to /content/drive/MyDrive/master_thesis/results/

=== E02_1 -- image-level summary ===
experiment_name  n_images  mAP50_mean  mAP50_std  precision_mean  precision_std  recall_mean  recall_std  IoU1_mean  IoU2_mean
          E02_1       136    0.630672   0.345089        0.459169       0.244108     0.659446    0.322301   0.819525   0.559239
